# (Opsiyonal) Olist reviews' - Translations...

* 🇧🇷 Brezilya Portekizcesi bilmiyorsanız, hadi `order_reviews` metinlerini 🇬🇧 İngilizce’ye çevirelim!

* Bunun için `pip install googletrans==4.0.2` yüklemeniz gerekecek.

☢️ Bu API ile herhangi bir sorun yaşarsanız, bunu düzeltmek için zaman harcamayın, bu package oldukça dengesiz… Aklınızda bulunsun:
- bu optional bir challenge
- Brezilya Portekizcesi ile yazılmış review’ların anlamını görmek için bazı yorumları Google Translate’e kopyalayıp yapıştırarak yine de eski yöntemle yapabilirsiniz.

## Review Translator

👉 `reviews` dataset’ini load edin ve 1-yıldız rating’e sahip review’lardan bir örnek (sample) seçin.

In [12]:
# Verileri yükle
from olist.data import Olist
data = Olist().get_data()

👀 20 adet düşük puan alan yorumdan oluşan bir örneklem seçin (rastgele) ve bunu bir listeye dönüştürün.

In [13]:
# OR
low_reviews_sample =\
    data['order_reviews'].dropna().query("review_score==1").sample(20)

low_reviews_sample_list=\
    low_reviews_sample['review_comment_message'].tolist()

🗣 Bu göreve başlamadan önce önceden yüklediğiniz `google_translator` paketini kullanarak bu yorumları çevirin:

In [25]:
from googletrans import Translator
import time


translator = Translator()
translations = []


for i, review in enumerate(low_reviews_sample_list):
    try:
        translation = await translator.translate(review, src='pt', dest='en')
        
        print(f"Yorum {i+1}:")
        print(translation.text)
        print("-" * 30) # Karışmaması için bir ayırıcı çizgi
        
        translations.append(translation.text)
        
        # Çok hızlı gidip engellenmemek için
        time.sleep(0.5)
        
    except Exception as e:
        print(f"Yorum {i+1} çevrilirken bir hata oluştu: {e}")
        translations.append("Translation Error")

print("\n--- İŞLEM TAMAMLANDI ---")

Yorum 1:
I only received the personalized thermos cup, but not the decorative plate.
------------------------------
Yorum 2:
I need to exchange the defective product and I cannot register the request on the SITE. In the item "EXCHANGES AND RETURNS" it always informs that there is a communication problem.
I await your return
------------------------------
Yorum 3:
I want to know why there is so much delay in my order, because they gave a delivery forecast for 4/23 and still nothing
------------------------------
Yorum 4:
I don't recommend it, in addition to the delay in delivery, only one chair arrived, and 2 were paid for. 
I would like to know if I can open a case, or will they send it to another department?
------------------------------
Yorum 5:
I DID NOT RECEIVE THE PRODUCTS MENTIONED IN THE EMAIL. I RECEIVED, AS I ALREADY SAID, AN EMPTY CARDBOARD BOX, WITHOUT ANY CONTENTS. AS IN THE OTHER MESSAGES I HAVE ALREADY SENT, I HOPE MY PROBLEM WILL BE RESOLVED.
---------------------------

In [27]:
import pandas as pd

# Listeleri bir DataFrame yapısında birleştiriyoruz
df_translated = pd.DataFrame({
    'original_portuguese': low_reviews_sample_list,
    'translated_english': translations
})

# CSV olarak kaydediyoruz
# encoding='utf-8-sig' kullanıyoruz ki Türkçe veya Portekizce karakterler Excel'de bozuk görünmesin
df_translated.to_csv('translated_reviews.csv', index=False, encoding='utf-8-sig')

print("İşlem başarılı! 'translated_reviews.csv' dosyası kaydedildi.")

# Kaydedilen dosyanın ilk birkaç satırını kontrol edelim
print(df_translated.head())

İşlem başarılı! 'translated_reviews.csv' dosyası kaydedildi.
                                 original_portuguese  \
0  Recebi apenas o copo térmico personalizado, ma...   
1  Preciso alizar a troca do produto que está com...   
2  Quero saber porque tanto atraso na minha encom...   
3  Não recomendo, além da demora para entregar, c...   
4  NÃO RECEBI OS PRODUTOS MENCIONADOS NO E-MAIL. ...   

                                  translated_english  
0  I only received the personalized thermos cup, ...  
1  I need to exchange the defective product and I...  
2  I want to know why there is so much delay in m...  
3  I don't recommend it, in addition to the delay...  
4  I DID NOT RECEIVE THE PRODUCTS MENTIONED IN TH...  


**Insights** 💡
- Bazı kötü review’lar delivery ile ilgili (`wait_time`, kaçırılan teslim tarihi, vb.)
- Ancak bazı kötü review’lar seller veya ürünle ilgili...

Peki iki olası nedeni nasıl ayırt edebiliriz?

Bu Olist’in mutlaka bilmesi gereken bir şey:
- bazı ürünleri katalogdan mı çıkarmalı?
- yoksa bazı seller’ları marketplace’ten mi kaldırmalı?
